In [1]:
import os 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 

In [2]:
sample = pd.read_csv("../data/raw/biospecimen/sample.tsv", sep="\t")

clinical = pd.read_csv("../data/raw/clinical/clinical.tsv", sep="\t")
follow_up = pd.read_csv("../data/raw/clinical/follow_up.tsv", sep="\t")
pathology = pd.read_csv("../data/raw/clinical/pathology_detail.tsv", sep="\t")

indian_liver_patient = pd.read_csv("../data/raw/indian_liver_patient.csv")
liver_disease = pd.read_csv("../data/raw/Liver_Disease_Patient.csv", encoding='latin-1')

### Clinical

In [3]:
df_clinical = clinical.copy()
df_clinical =df_clinical.replace("--", np.nan)
df_clinical=df_clinical.replace("--", np.nan)


clinical_new_cols=[
    "cases.case_id",
    "demographic.age_at_index",
    "demographic.gender",
    "demographic.race" ,
    "demographic.vital_status",
    "demographic.days_to_death",
    "diagnoses.age_at_diagnosis",
    "diagnoses.primary_diagnosis",
    "diagnoses.ajcc_pathologic_stage",
    "diagnoses.tumor_grade",
    "diagnoses.year_of_diagnosis",
]



clinical_new_cols=[c for c in clinical_new_cols if c in clinical.columns]
df_clinical = clinical[clinical_new_cols].copy()
df_clinical.columns = [c.split(".")[-1] for c in df_clinical.columns]

print(f"Shape: {df_clinical.shape}")
print("Columns:", df_clinical.columns.tolist())

df_clinical.head(3)


Shape: (2977, 11)
Columns: ['case_id', 'age_at_index', 'gender', 'race', 'vital_status', 'days_to_death', 'age_at_diagnosis', 'primary_diagnosis', 'ajcc_pathologic_stage', 'tumor_grade', 'year_of_diagnosis']


,case_id,age_at_index,gender,race,vital_status,days_to_death,age_at_diagnosis,primary_diagnosis,ajcc_pathologic_stage,tumor_grade,year_of_diagnosis
0,0004d251-3f70-4395-b175-c94c2f5b1b81,48,male,asian,Alive,'--,17833,"Hepatocellular carcinoma, NOS",Stage I,G1,2007
1,0004d251-3f70-4395-b175-c94c2f5b1b81,48,male,asian,Alive,'--,17833,"Hepatocellular carcinoma, NOS",Stage I,G1,2007
2,0004d251-3f70-4395-b175-c94c2f5b1b81,48,male,asian,Alive,'--,17833,"Hepatocellular carcinoma, NOS",Stage I,G1,2007


### Target 1 :Risk 

In [4]:
print (df_clinical["vital_status"].value_counts())
df_clinical["target_risk"] = df_clinical["vital_status"].map({"Alive": 0, "Dead": 1 , "alive": 0, "dead": 1,"ALIVE": 0, "DEAD": 1  })

df_clinical["target_risk"]= df_clinical["target_risk"].fillna(0).astype(int)
print("Target Head -Risk:")
print(df_clinical["target_risk"].value_counts())

vital_status
Alive    1841
Dead     1136
Name: count, dtype: int64
Target Head -Risk:
target_risk
0    1841
1    1136
Name: count, dtype: int64


### Target 2:Stage 

In [5]:
print(df_clinical["ajcc_pathologic_stage"])

0       Stage I
1       Stage I
2       Stage I
3       Stage I
4           '--
         ...   
2972    Stage I
2973    Stage I
2974    Stage I
2975    Stage I
2976    Stage I
Name: ajcc_pathologic_stage, Length: 2977, dtype: object


In [6]:
print(df_clinical["ajcc_pathologic_stage"].value_counts())

stage_mapping = {
    "Stage I": 0, "Stage IA": 0, "Stage IB": 0,
    "Stage II":1,
    "Stage III": 2, "Stage IIIA": 2, "Stage IIIB": 2, "Stage IIIC": 2,
    "Stage IV": 3, "Stage IVA": 3, "Stage IVB": 3, "Stage IVC": 3
}

df_clinical["target_stage"] = df_clinical["ajcc_pathologic_stage"].map(stage_mapping)
mode_stage= df_clinical["target_stage"].mode()
df_clinical["target_stage"] = df_clinical["target_stage"].fillna(mode_stage.iloc[0] if len(mode_stage) > 0 else 0).astype(int)

print("Target Head -Stage:")
print(df_clinical["target_stage"].value_counts())

ajcc_pathologic_stage
'--           1387
Stage I        781
Stage II       397
Stage IIIA     278
Stage IIIC      43
Stage IIIB      37
Stage III       21
Stage IV        11
Stage IVB       10
Stage IIB        5
Stage IVA        4
Stage 0          3
Name: count, dtype: int64
Target Head -Stage:
target_stage
0    2176
1     397
2     379
3      25
Name: count, dtype: int64


### Target 3:Severity

In [7]:
print(df_clinical["tumor_grade"].value_counts())
df_clinical["tumor_grade"].head(10)

several_mapping = {
    "G1": 0, "G2": 1, "G3": 2, "G4": 2,
    "Low Grade": 0, "High Grade": 2,
    "Well differentiated": 0, "Moderately differentiated": 1, 
    "Poorly differentiated": 2, "Undifferentiated": 2

}
df_clinical["target_severity"] = df_clinical["tumor_grade"].map(several_mapping)

if df_clinical["target_severity"].isna().mean() > df_clinical.shape[0] * 0.5:
    df_clinical["target_severity"] = df_clinical["target_stage"].apply(
        lambda x:0 if x == 0 else (1 if x == 1 else 2))
else:
    df_clinical["target_severity"] = df_clinical["target_severity"].fillna(1).astype(int)

print ("Target Head -Severity:")
print(df_clinical["target_severity"].value_counts())


tumor_grade
'--    1351
G2      774
G3      567
G1      230
G4       55
Name: count, dtype: int64
Target Head -Severity:
target_severity
1    2125
2     622
0     230
Name: count, dtype: int64


In [8]:
df_clinical["gender"] =df_clinical["gender"].map({
    "male":0,
    "female":1,
    "MALE" :0,
    "FEMALE":1
})

gender_mode =df_clinical["gender"].mode()
df_clinical["gender"] = df_clinical["gender"].fillna(gender_mode.iloc[0] if len(gender_mode) > 0 else 0)

df_clinical["age_at_index"]=pd.to_numeric(df_clinical["age_at_index"], errors="coerce")
df_clinical["age_at_index"] = df_clinical["age_at_index"].fillna(df_clinical["age_at_index"].median())

df_clinical["age_at_diagnosis"]=pd.to_numeric(df_clinical["age_at_diagnosis"], errors="coerce")
df_clinical["age_at_diagnosis"] = (df_clinical["age_at_diagnosis"] / 365.25).round(1)
df_clinical["age_at_diagnosis"] = df_clinical["age_at_diagnosis"].fillna(df_clinical["age_at_diagnosis"].median())


df_clinical["race"]=df_clinical["race"].fillna(df_clinical["race"].mode().iloc[0] if len( df_clinical["race"].mode()) > 0 else "Unknown")

print("Cleaned Clinical Data")

print("Total null values in df_clinical:", df_clinical.isnull().sum().sum())

df_clinical.shape

Cleaned Clinical Data
Total null values in df_clinical: 0


(2977, 14)

In [9]:
df_clinical.isnull().sum()

case_id                  0
age_at_index             0
gender                   0
race                     0
vital_status             0
days_to_death            0
age_at_diagnosis         0
primary_diagnosis        0
ajcc_pathologic_stage    0
tumor_grade              0
year_of_diagnosis        0
target_risk              0
target_stage             0
target_severity          0
dtype: int64

### Follow-Up

In [10]:
follow_up=follow_up.replace("--", np.nan)

follow_up_cols=[i for i in follow_up.columns if any (k in i.lower() for k in ["case_id", "progression_or_recurrence", "days_to_progression"])]

df_follow_up = follow_up[follow_up_cols].copy()
df_follow_up.columns = [c.split(".")[-1] for c in df_follow_up.columns]
df_follow_up=df_follow_up.iloc[:, ~df_follow_up.columns.duplicated()]

print(f"Shape: {df_follow_up.shape}")
print("Columns:", df_follow_up.columns.tolist())

if "progression_or_recurrence" in df_follow_up.columns:
    df_follow_up["progression_or_recurrence"] = df_follow_up["progression_or_recurrence"].map({
        "Yes": 1, "No": 0, "YES": 1, "NO": 0
    })
    df_follow_up["progression_or_recurrence"] = df_follow_up["progression_or_recurrence"].fillna(0).astype(int)

    print("Progression or Recurrence Value Counts:")
    print(df_follow_up["progression_or_recurrence"].value_counts())

for col in ["days_to_progression" , "days_to_progression_free"]:
    if col in df_follow_up.columns:
        df_follow_up[col] = pd.to_numeric(df_follow_up[col], errors="coerce")
        df_follow_up[col] = df_follow_up[col].fillna(df_follow_up[col].median())

print(f"Missing values: {df_follow_up.isnull().sum().sum()}")
print(f"Shape: {df_follow_up.shape}")

df_follow_up.head(3)

Shape: (4811, 6)
Columns: ['case_id', 'days_to_progression', 'days_to_progression_free', 'progression_or_recurrence', 'progression_or_recurrence_anatomic_site', 'progression_or_recurrence_type']
Progression or Recurrence Value Counts:
progression_or_recurrence
0    4576
1     235
Name: count, dtype: int64
Missing values: 9622
Shape: (4811, 6)


c:\Users\islem\OneDrive\Desktop\pjresearch_model\env\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\islem\OneDrive\Desktop\pjresearch_model\env\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,case_id,days_to_progression,days_to_progression_free,progression_or_recurrence,progression_or_recurrence_anatomic_site,progression_or_recurrence_type
0,0004d251-3f70-4395-b175-c94c2f5b1b81,NaN,NaN,1,"Lung, NOS",Unknown
1,0004d251-3f70-4395-b175-c94c2f5b1b81,NaN,NaN,1,Liver,Local
2,0004d251-3f70-4395-b175-c94c2f5b1b81,NaN,NaN,0,'--,'--


### Pathology

In [11]:
pathology =pathology.replace("--", np.nan)

pathology_cols=[i for i in pathology.columns if any (k in i.lower() for k in ["case_id", "tumor", "necrosis", "vascular","lymph", "margin",
    "greatest_tumor", "residual", "invasion"])]

df_pathology = pathology[pathology_cols].copy()
df_pathology.columns = [c.split(".")[-1] for c in df_pathology.columns]
df_pathology=df_pathology.iloc[:, ~df_pathology.columns.duplicated()]

print(f"Shape: {df_pathology.shape}")
print("Columns:", df_pathology.columns.tolist())

Shape: (377, 37)
Columns: ['case_id', 'circumferential_resection_margin', 'greatest_tumor_dimension', 'gross_tumor_weight', 'lymph_node_dissection_method', 'lymph_node_dissection_site', 'lymph_node_involved_site', 'lymph_node_involvement', 'lymph_nodes_positive', 'lymph_nodes_removed', 'lymph_nodes_tested', 'lymphatic_invasion_present', 'margin_status', 'necrosis_percent', 'necrosis_present', 'non_nodal_tumor_deposits', 'percent_tumor_invasion', 'percent_tumor_nuclei', 'perineural_invasion_present', 'peripancreatic_lymph_nodes_positive', 'peripancreatic_lymph_nodes_tested', 'residual_tumor', 'residual_tumor_measurement', 'tumor_basal_diameter', 'tumor_burden', 'tumor_depth_descriptor', 'tumor_depth_measurement', 'tumor_infiltrating_lymphocytes', 'tumor_infiltrating_macrophages', 'tumor_largest_dimension_diameter', 'tumor_length_measurement', 'tumor_level_prostate', 'tumor_shape', 'tumor_thickness', 'tumor_width_measurement', 'vascular_invasion_present', 'vascular_invasion_type']


In [12]:
binary_cols =[i for i in df_pathology.columns if df_pathology[i].dropna().isin(["Yes", "No", "YES", "NO"]).all()]
for col in binary_cols:
    df_pathology[col] = df_pathology[col].map({
        "Yes": 1, "No": 0, "YES": 1, "NO": 0
    })
    df_pathology[col] = df_pathology[col].fillna(0).astype(int)

print('Missing values:', df_pathology.isnull().sum().sum())
print(f"Shape: {df_pathology.shape}")

Missing values: 0
Shape: (377, 37)


### Biospecimen

In [13]:
sample = sample.replace("--", np.nan)

sample_cols=[i for i in sample.columns if any (k in i.lower() for k in ["case_id", "sample_type", "tissue_type", "tumor_descriptor"])]

df_sample = sample[sample_cols].copy()
df_sample.columns = [c.split(".")[-1] for c in df_sample.columns]
df_sample=df_sample.iloc[:, ~df_sample.columns.duplicated()]

print(f"Shape: {df_sample.shape}")
print("Columns:", df_sample.columns.tolist())

Shape: (1168, 4)
Columns: ['case_id', 'sample_type', 'tissue_type', 'tumor_descriptor']


In [14]:
if "sample_type"in df_sample.columns:
    df_sample['is_tumor']= df_sample['sample_type'].apply(lambda x: 1 if isinstance(x, str) and "tumor" in x.lower() else 0)

if "tissue_type" in df_sample.columns:
    df_sample['is_tumor_tissue']= df_sample['tissue_type'].apply(lambda x: 1 if isinstance(x, str) and "tumor" in x.lower() else 0)

if "tumor_descriptor" in df_sample.columns:
    df_sample["is_primary_tumor"] = df_sample["tumor_descriptor"].apply(lambda x: 1 if isinstance(x, str) and "primary" in x.lower() else 0)

df_sample = df_sample.fillna(0)

print(f"Missing values: {df_sample.isnull().sum().sum()}")
print(f"Shape: {df_sample.shape}")
df_sample.head(3)

Missing values: 0
Shape: (1168, 7)


,case_id,sample_type,tissue_type,tumor_descriptor,is_tumor,is_tumor_tissue,is_primary_tumor
0,834cff01-3c12-42fa-abbe-ae0ae168c568,Blood Derived Normal,Normal,Not Applicable,0,0,0
1,834cff01-3c12-42fa-abbe-ae0ae168c568,Primary Tumor,Tumor,Primary,1,1,1
2,834cff01-3c12-42fa-abbe-ae0ae168c568,Primary Tumor,Tumor,Primary,1,1,1


### Merge All files 

In [15]:
def rename_case_id(df):
    for col in df.columns:
        if "case_id" in col.lower():
            df = df.rename(columns={col: "case_id"})
            break
    return df

df_clinical = rename_case_id(df_clinical)
df_follow_up = rename_case_id(df_follow_up)
df_sample = rename_case_id(df_sample)

df_tcga =df_clinical.copy()
print(f"Initial shape of df_tcga: {df_tcga.shape}")



Initial shape of df_tcga: (2977, 14)


In [16]:
print(df_tcga.columns.tolist())

['case_id', 'age_at_index', 'gender', 'race', 'vital_status', 'days_to_death', 'age_at_diagnosis', 'primary_diagnosis', 'ajcc_pathologic_stage', 'tumor_grade', 'year_of_diagnosis', 'target_risk', 'target_stage', 'target_severity']


In [17]:
follow_up_merged_cols =["case_id"] + [col for col in df_follow_up.columns if col != "case_id"
                        and col in ["progression_or_recurrence", "days_to_progression", "days_to_progression_free"]]
follow_up_merged_cols=[col for col in follow_up_merged_cols if col in df_follow_up.columns]

df_tcga = df_tcga.merge(df_follow_up[follow_up_merged_cols].drop_duplicates("case_id"), on="case_id", how="left")
print(f"Shape after merging follow_up: {df_tcga.shape}")


sample_merged_cols = ["case_id"] + [col for col in df_sample.columns if col != "case_id" and col in ["is_tumor", "is_tumor_tissue","is_primary_tumor"]]
sample_merged_cols = [col for col in sample_merged_cols if col in df_sample.columns]

df_tcga = df_tcga.merge(df_sample[sample_merged_cols].drop_duplicates("case_id"), on="case_id", how="left")
print(f"Shape after merging sample: {df_tcga.shape}")


for col in df_tcga.select_dtypes(include=np.number).columns:
    df_tcga[col] = df_tcga[col].fillna(df_tcga[col].median())

print(f"Final shape of df_tcga: {df_tcga.shape}")
print(f"Total missing values in df_tcga: {df_tcga.isnull().sum().sum()}")
df_tcga.head(3)

Shape after merging follow_up: (2977, 17)
Shape after merging sample: (2977, 20)
Final shape of df_tcga: (2977, 20)
Total missing values in df_tcga: 5954


c:\Users\islem\OneDrive\Desktop\pjresearch_model\env\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\islem\OneDrive\Desktop\pjresearch_model\env\lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,case_id,age_at_index,gender,race,vital_status,days_to_death,age_at_diagnosis,primary_diagnosis,ajcc_pathologic_stage,tumor_grade,year_of_diagnosis,target_risk,target_stage,target_severity,days_to_progression,days_to_progression_free,progression_or_recurrence,is_tumor,is_tumor_tissue,is_primary_tumor
0,0004d251-3f70-4395-b175-c94c2f5b1b81,48.0,0,asian,Alive,'--,48.8,"Hepatocellular carcinoma, NOS",Stage I,G1,2007,0,0,0,NaN,NaN,1,1,1,1
1,0004d251-3f70-4395-b175-c94c2f5b1b81,48.0,0,asian,Alive,'--,48.8,"Hepatocellular carcinoma, NOS",Stage I,G1,2007,0,0,0,NaN,NaN,1,1,1,1
2,0004d251-3f70-4395-b175-c94c2f5b1b81,48.0,0,asian,Alive,'--,48.8,"Hepatocellular carcinoma, NOS",Stage I,G1,2007,0,0,0,NaN,NaN,1,1,1,1


In [18]:
print("Targets :")
print(f"Risk: {df_tcga['target_risk'].value_counts().to_string()}")
print(f"Stage: {df_tcga['target_stage'].value_counts().to_string()}")
print(f"Severity: {df_tcga['target_severity'].value_counts().to_string()}")

df_tcga.head(3)

Targets :
Risk: target_risk
0    1841
1    1136
Stage: target_stage
0    2176
1     397
2     379
3      25
Severity: target_severity
1    2125
2     622
0     230


,case_id,age_at_index,gender,race,vital_status,days_to_death,age_at_diagnosis,primary_diagnosis,ajcc_pathologic_stage,tumor_grade,year_of_diagnosis,target_risk,target_stage,target_severity,days_to_progression,days_to_progression_free,progression_or_recurrence,is_tumor,is_tumor_tissue,is_primary_tumor
0,0004d251-3f70-4395-b175-c94c2f5b1b81,48.0,0,asian,Alive,'--,48.8,"Hepatocellular carcinoma, NOS",Stage I,G1,2007,0,0,0,NaN,NaN,1,1,1,1
1,0004d251-3f70-4395-b175-c94c2f5b1b81,48.0,0,asian,Alive,'--,48.8,"Hepatocellular carcinoma, NOS",Stage I,G1,2007,0,0,0,NaN,NaN,1,1,1,1
2,0004d251-3f70-4395-b175-c94c2f5b1b81,48.0,0,asian,Alive,'--,48.8,"Hepatocellular carcinoma, NOS",Stage I,G1,2007,0,0,0,NaN,NaN,1,1,1,1


### Indian Liver Dataset

In [19]:
df_indian = indian_liver_patient.copy()

df_indian ["Dataset"] = df_indian["Dataset"].map({1:1,2:0})
df_indian.rename(columns={"Dataset": "target_risk"}, inplace=True)

df_indian["Gender"]= df_indian["Gender"].map({"Male":1,"Female":0})
df_indian["Gender"]= df_indian["Gender"].fillna(df_indian["Gender"].mode().iloc[0])


df_indian["Albumin_and_Globulin_Ratio"] = df_indian["Albumin_and_Globulin_Ratio"].fillna(df_indian["Albumin_and_Globulin_Ratio"].median())

# Add stage and severity from risk
df_indian["target_stage"]= df_indian["target_risk"].apply(lambda x: 1 if x == 1 else 0)
df_indian["target_severity"] = df_indian["target_risk"].apply(lambda x: 1 if x == 1 else 0)

print(f"Indian Liver Patient Shape : {df_indian.shape}")
print(f"Missing values in Indian Liver Patient: {df_indian.isnull().sum().sum()}    ")
print("Targets in Indian Liver Patient:")
print(df_indian["target_risk"].value_counts())

Indian Liver Patient Shape : (583, 13)
Missing values in Indian Liver Patient: 0    
Targets in Indian Liver Patient:
target_risk
1    416
0    167
Name: count, dtype: int64


### Liver Disease Dataset

In [20]:
df_liver_disease = liver_disease.copy()

# Clean column names
df_liver_disease.columns = (df_liver_disease.columns.str.strip().str.replace(r"[^\w]", "_", regex=True).str.replace(r"_+", "_", regex=True).str.strip("_"))

print(f"Columns: {df_liver_disease.columns.tolist()}")

# Rename target column → target_risk
df_liver_disease = df_liver_disease.rename(columns={"Result": "target_risk"})

# Encode target: 1=Disease, 2=Healthy → 1=Disease, 0=Healthy
df_liver_disease["target_risk"] = df_liver_disease["target_risk"].map({1: 1, 2: 0})

print(f"Target distribution:")
print(df_liver_disease["target_risk"].value_counts())

# Encode gender
gender_col = [c for c in df_liver_disease.columns if "gender" in c.lower()]
if gender_col:
    df_liver_disease[gender_col[0]] = df_liver_disease[gender_col[0]].map({"Male": 1, "Female": 0, "male": 1, "female": 0})
    df_liver_disease[gender_col[0]] = df_liver_disease[gender_col[0]].fillna(df_liver_disease[gender_col[0]].mode().iloc[0]if len(df_liver_disease[gender_col[0]].mode()) > 0 else 0)

# Fill missing values
for col in df_liver_disease.select_dtypes(include=np.number).columns:
    df_liver_disease[col] = df_liver_disease[col].fillna(df_liver_disease[col].median())

# Add stage and severity
df_liver_disease["target_stage"]= df_liver_disease["target_risk"].apply(lambda x: 1 if x == 1 else 0)
df_liver_disease["target_severity"] = df_liver_disease["target_risk"].apply(lambda x: 1 if x == 1 else 0)

print(f"\nShape: {df_liver_disease.shape}")
print(f"Missing values: {df_liver_disease.isnull().sum().sum()}")
print(f"\nTarget distribution:")
print(df_liver_disease["target_risk"].value_counts())

Columns: ['Age_of_the_patient', 'Gender_of_the_patient', 'Total_Bilirubin', 'Direct_Bilirubin', 'Alkphos_Alkaline_Phosphotase', 'Sgpt_Alamine_Aminotransferase', 'Sgot_Aspartate_Aminotransferase', 'Total_Protiens', 'ALB_Albumin', 'A_G_Ratio_Albumin_and_Globulin_Ratio', 'Result']
Target distribution:
target_risk
1    21917
0     8774
Name: count, dtype: int64

Shape: (30691, 13)
Missing values: 0

Target distribution:
target_risk
1    21917
0     8774
Name: count, dtype: int64


### SMOTE

In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "imbalanced-learn", "-q"])
    from imblearn.over_sampling import SMOTE

In [22]:
df_main = df_indian.copy()

target_cols = ["target_risk", "target_stage", "target_severity"]
X = df_main.select_dtypes(include=np.number).drop(columns=target_cols, errors="ignore")
y_risk= df_main["target_risk"].astype(int)
y_stage= df_main["target_stage"].astype(int)
y_severity = df_main["target_severity"].astype(int)

print(f"Features: {X.columns.tolist()}")
print(f"X shape: {X.shape}")
print(f"Class distribution (risk): {y_risk.value_counts().to_dict()}")

Features: ['Age', 'Gender', 'Total_Bilirubin', 'Direct_Bilirubin', 'Alkaline_Phosphotase', 'Alamine_Aminotransferase', 'Aspartate_Aminotransferase', 'Total_Protiens', 'Albumin', 'Albumin_and_Globulin_Ratio']
X shape: (583, 10)
Class distribution (risk): {1: 416, 0: 167}


In [23]:
smote = SMOTE(random_state=42)
X_resampled, y_risk_resampled = smote.fit_resample(X, y_risk)

print(f"Before SMOTE: {X.shape} | {y_risk.value_counts().to_dict()}")
print(f"After  SMOTE: {X_resampled.shape} | {pd.Series(y_risk_resampled).value_counts().to_dict()}")

Before SMOTE: (583, 10) | {1: 416, 0: 167}
After  SMOTE: (832, 10) | {1: 416, 0: 416}


### Scaling

In [24]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_resampled)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f"Scaled shape: {X_scaled.shape}")
print(f"Mean (should be ~0): {X_scaled.mean().round(2).values}")
print(f"Std  (should be ~1): {X_scaled.std().round(2).values}")

Scaled shape: (832, 10)
Mean (should be ~0): [-0.  0. -0. -0.  0.  0. -0.  0. -0.  0.]
Std  (should be ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### Split

In [25]:
X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y_risk_resampled,test_size=0.15, random_state=42,stratify=y_risk_resampled)

X_train, X_val, y_train, y_val = train_test_split( X_temp, y_temp,test_size=0.176, random_state=42,stratify=y_temp)

print(f"Train: {X_train.shape}| {pd.Series(y_train).value_counts().to_dict()}")
print(f"Val: {X_val.shape}| {pd.Series(y_val).value_counts().to_dict()}")
print(f"Test: {X_test.shape}| {pd.Series(y_test).value_counts().to_dict()}")

Train: (582, 10)| {0: 291, 1: 291}
Val: (125, 10)| {1: 63, 0: 62}
Test: (125, 10)| {0: 63, 1: 62}


In [30]:
import os
import pickle

SAVE_PATH = r"C:\Users\islem\OneDrive\Desktop\pjresearch_model\data\processed"

os.makedirs(SAVE_PATH, exist_ok=True)

In [31]:
import pickle

# Save train/val/test splits
X_train.to_csv(f"{SAVE_PATH}/X_train.csv", index=False)
X_val.to_csv(f"{SAVE_PATH}/X_val.csv",     index=False)
X_test.to_csv(f"{SAVE_PATH}/X_test.csv",   index=False)

pd.Series(y_train).to_csv(f"{SAVE_PATH}/y_train.csv", index=False)
pd.Series(y_val).to_csv(f"{SAVE_PATH}/y_val.csv",     index=False)
pd.Series(y_test).to_csv(f"{SAVE_PATH}/y_test.csv",   index=False)

# Save scaler
with open(f"{SAVE_PATH}/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save processed datasets
df_tcga.to_csv(f"{SAVE_PATH}/tcga_merged.csv",      index=False)
df_indian.to_csv(f"{SAVE_PATH}/indian_processed.csv", index=False)
df_liver_disease.to_csv(f"{SAVE_PATH}/df_liver_disease_processed.csv",      index=False)

print("All files saved!")
print(f"Files: {os.listdir(SAVE_PATH)}")

All files saved!
Files: ['df_liver_disease_processed.csv', 'indian_processed.csv', 'scaler.pkl', 'tcga_merged.csv', 'X_test.csv', 'X_train.csv', 'X_val.csv', 'y_test.csv', 'y_train.csv', 'y_val.csv']


In [32]:
print("="*55)
print("    HepaXAI — PREPROCESSING SUMMARY")
print("="*55)
print(f"TCGA Merged      : {df_tcga.shape}")
print(f"Indian Processed : {df_indian.shape}")
print(f"Liver Patient Disease Processed    : {df_liver_disease.shape}")
print(f"Training splits:")
print(f"X_train : {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}")
print(f"Features: {X_train.columns.tolist()}")
print(f"Targets:")
print(f"Head 1 (Risk)     : binary (0=Healthy, 1=Disease)")
print(f"Head 2 (Stage)    : multiclass (0=I, 1=II, 2=III, 3=IV)")
print(f"Head 3 (Severity) : multiclass (0=Mild, 1=Moderate, 2=Severe)")
print("="*55)


    HepaXAI — PREPROCESSING SUMMARY
TCGA Merged      : (2977, 20)
Indian Processed : (583, 13)
Liver Patient Disease Processed    : (30691, 13)
Training splits:
X_train : (582, 10)
X_val: (125, 10)
X_test: (125, 10)
Features: ['Age', 'Gender', 'Total_Bilirubin', 'Direct_Bilirubin', 'Alkaline_Phosphotase', 'Alamine_Aminotransferase', 'Aspartate_Aminotransferase', 'Total_Protiens', 'Albumin', 'Albumin_and_Globulin_Ratio']
Targets:
Head 1 (Risk)     : binary (0=Healthy, 1=Disease)
Head 2 (Stage)    : multiclass (0=I, 1=II, 2=III, 3=IV)
Head 3 (Severity) : multiclass (0=Mild, 1=Moderate, 2=Severe)
